# Week 03: Break the Spam Filter

## Objective

The goal of this project is to train a simple SMS spam classifier and investigate how easily carefully written messages can fool it.

## Model

We use TF-IDF to convert text into numerical features and Multinomial Naive Bayes to classify messages as spam or ham.

## Experiments

1. Load the SMS Spam Collection dataset.
2. Split the data into training and testing sets.
3. Train the classifier and measure accuracy.
4. Test five carefully designed messages.
5. Record the predictions and analyze why the model may fail.

In [1]:
%pip install -q pandas scikit-learn

In [2]:
import os
import zipfile
import urllib.request
from pathlib import Path

import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
DATA_URL = (
    "https://archive.ics.uci.edu/static/public/228/"
    "sms+spam+collection.zip"
)

ZIP_PATH = Path("sms_spam_collection.zip")
DATA_DIR = Path("sms_spam_collection")
DATA_FILE = DATA_DIR / "SMSSpamCollection"

if not DATA_FILE.exists():
    DATA_DIR.mkdir(parents=True, exist_ok=True)

    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)

    with zipfile.ZipFile(ZIP_PATH, "r") as archive:
        archive.extractall(DATA_DIR)

print("Dataset ready:", DATA_FILE)
print("Dataset file exists:", DATA_FILE.exists())

Dataset ready: sms_spam_collection/SMSSpamCollection
Dataset file exists: True


In [4]:
df = pd.read_csv(
    DATA_FILE,
    sep="\t",
    header=None,
    names=["label", "message"],
    encoding="utf-8",
)

print("Dataset shape:", df.shape)
print("\nLabel distribution:")
print(df["label"].value_counts())

display(df.head())

Dataset shape: (5572, 2)

Label distribution:
label
ham     4825
spam     747
Name: count, dtype: int64


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [5]:
emails = df["message"]
labels = df["label"]

print("Total messages:", len(emails))
print("Total labels:", len(labels))
print("Classes:", labels.unique())

Total messages: 5572
Total labels: 5572
Classes: ['ham' 'spam']


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    emails, labels, test_size=0.2, random_state=42
)

vec = TfidfVectorizer()
model = MultinomialNB()

model.fit(vec.fit_transform(X_train), y_train)

y_pred = model.predict(vec.transform(X_test))

accuracy = accuracy_score(y_test, y_pred)

print(f"Test accuracy: {accuracy:.4%}")

Test accuracy: 96.6816%


In [7]:
print("Classification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Classification Report:
              precision    recall  f1-score   support

         ham       0.96      1.00      0.98       966
        spam       1.00      0.75      0.86       149

    accuracy                           0.97      1115
   macro avg       0.98      0.88      0.92      1115
weighted avg       0.97      0.97      0.96      1115

Confusion Matrix:
[[966   0]
 [ 37 112]]


## Testing Messages That May Fool the Classifier

We test five messages designed to challenge the classifier using spelling substitutions, negation, businesslike language, and misleading keywords.

The actual predictions will be recorded after running the model. A prediction is considered a successful evasion when a spam message is classified as ham, or a false alarm when a legitimate message is classified as spam.

In [8]:
tests = [
    "Cl@im your fr33 g1ft card now",
    "Not free, just confirming lunch at 1pm",
    "Congratulations! You have won a free prize. Claim now!",
    "Your account review is complete. Please verify your details.",
    "Hey, I am not selling anything. Are we still meeting tomorrow?",
]

predictions = model.predict(vec.transform(tests))

results = pd.DataFrame({
    "My message": tests,
    "Model said": predictions,
})

display(results)

,My message,Model said
0,Cl@im your fr33 g1ft card now,ham
1,"Not free, just confirming lunch at 1pm",ham
2,Congratulations! You have won a free prize. Cl...,spam
3,Your account review is complete. Please verify...,ham
4,"Hey, I am not selling anything. Are we still m...",ham


In [9]:
intended_labels = [
    "spam",
    "ham",
    "spam",
    "spam",
    "ham",
]

results["Should be"] = intended_labels

def explain_failure(row):
    if row["Model said"] == row["Should be"]:
        return "Correct prediction; no failure on this example."

    return "Misclassification; inspect wording and learned features."

results["Why it failed"] = results.apply(explain_failure, axis=1)

display(results[["My message", "Model said", "Should be", "Why it failed"]])

,My message,Model said,Should be,Why it failed
0,Cl@im your fr33 g1ft card now,ham,spam,Misclassification; inspect wording and learned...
1,"Not free, just confirming lunch at 1pm",ham,ham,Correct prediction; no failure on this example.
2,Congratulations! You have won a free prize. Cl...,spam,spam,Correct prediction; no failure on this example.
3,Your account review is complete. Please verify...,ham,spam,Misclassification; inspect wording and learned...
4,"Hey, I am not selling anything. Are we still m...",ham,ham,Correct prediction; no failure on this example.


## What I Learned

This experiment showed me that a spam classifier can achieve good test accuracy while still making mistakes on carefully written messages. I also learned that spelling tricks, negation, and misleading context can expose limitations in a model that relies on TF-IDF features and Naive Bayes.